# Tiếp tục Lab Day10 — E3/E4 đủ300 bước
Phan Trọng Hoàn · Colab T4 · chỉ dùng GPU khi chạy model. Giữ nguyên bài đã nộp tại commit9e211e9: kết quả200 bước trong report/evidence và REPORT.md chưa bị thay thế. Kết quả mới ghi vào **report/extended/**, run mới tên e3_r300_v1 / e4_data10_r300_v1. Chạy từng chặng, không Run all.

Mục tiêu trước hết: smoke pause/resume 4→8 bước rồi E3 đủ300. Checkpoint200 cũ không có optimizer/RNG nên **không resume được chính xác**. GPU chỉ xác nhận được khi người dùng chạy notebook này; kiểm tra CPU local không thay thế smoke T4.

## 0. Upload source mới và môi trường
Mở notebook này bằng Upload notebook; chọn T4 GPU. Upload **work/colab_resume_bundle.zip** được chuẩn bị trên local. Nếu /content/vlm_lab300 đã có, bỏ qua upload và tiếp tục cell setup.

In [ ]:
from pathlib import Path
from zipfile import ZipFile
from google.colab import files
ROOT = Path("/content/vlm_lab300")
if ROOT.exists():
    raise FileExistsError("Workspace đã có; bỏ qua upload, không giải nén đè.")
uploaded = files.upload()
assert len(uploaded) == 1, "Chọn một ZIP source mới."
with ZipFile(next(iter(uploaded))) as z:
    for item in z.infolist():
        assert (ROOT / item.filename).resolve().is_relative_to(ROOT.resolve()), item.filename
    z.extractall(ROOT)
print(ROOT)

In [ ]:
%cd /content/vlm_lab300
import sys
from pathlib import Path
from src.colab_workflow import run
run("-m", "pip", "install", "-r", "requirements-colab.txt")
run("-m", "pip", "install", "pytest")
import importlib.metadata as metadata
from packaging.version import Version
try:
    torchao_version = metadata.version("torchao")
except metadata.PackageNotFoundError:
    torchao_version = None
if torchao_version is not None and Version(torchao_version) < Version("0.16.0"):
    run("-m", "pip", "uninstall", "-y", "torchao")
# Nếu Colab yêu cầu restart, chạy lại cell này; bỏ qua upload.

In [ ]:
import json
import shutil
import torch
from google.colab import drive
assert torch.cuda.is_available(), "Cần Colab T4 GPU."
print("GPU:", torch.cuda.get_device_name(0), "torch:", torch.__version__)
for name in ("transformers", "peft", "bitsandbytes", "accelerate"):
    print(name, metadata.version(name))
drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/day10_vlm")
BACKUP = str(DRIVE / "runs")
Path(BACKUP).mkdir(parents=True, exist_ok=True)
RESULTS = Path("report/extended")
(RESULTS / "evidence").mkdir(parents=True, exist_ok=True)
(RESULTS / "validation").mkdir(parents=True, exist_ok=True)
print("Drive free per mount (GiB; đối chiếu thêm UI Drive):", round(shutil.disk_usage(DRIVE).free/1024**3, 2))

## 1. Khôi phục baseline và dataset
E1/E2 đủ300 có thể dùng làm đối chứng; không train lại. Khôi phục trọng số giữa Drive và Colab, không tải về Git. Chỉ cần E1/E2 checkpoint sau khi chọn đối chứng, nhưng cell dưới lấy sẵn cả hai. report/extended chứa bản sao baseline, không ghi đè evidence đã nộp.

In [ ]:
from src.colab_workflow import restore_from_drive
restore_from_drive(DRIVE, ["e1", "e2"])
for name in ("e0", "e1", "e2"):
    target = RESULTS / "evidence" / f"{name}.json"
    if not target.exists():
        shutil.copy2(f"report/evidence/{name}.json", target)
for name in ("e1", "e2"):
    target = RESULTS / "validation" / f"{name}.json"
    if not target.exists():
        shutil.copy2(f"report/validation/{name}.json", target)
    run_meta = json.loads(Path(f"work/runs/{name}/step-0300/run.json").read_text())
    assert run_meta["steps"] == 300 and not run_meta["smoke"]

In [ ]:
import os
if not Path("data/splits/meta.json").exists():
    from google.colab import userdata
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
    run("scripts/prepare_data.py", "--kaggle")
from src.data import inspect_data
meta = inspect_data("data")
assert meta["dataset_version"] == "d10-21e4070301"
# Nếu dataset đã có tại workspace cũ cùng runtime, có thể chép data từ /content/vlm_lab
# trước cell này để tránh tải lại; không đổi split.

## 2. Smoke mới E3 — pause4 rồi resume8
Smoke kiểm tra state optimizer/scheduler/GradScaler/RNG/batch cursor và checksum; marker cũ200 không dùng được. Bản ZIP resume mới gồm adapter + projector + training_state.pt. Thao tác resume T4 vẫn cần xem log thực tế, chưa được xác nhận ở local.

In [ ]:
CFG = "configs/e3.yaml"
E3_ARGS = ["--config", CFG, "--amp-init-scale", "1024"]
E3_SMOKE = "work/runs/e3_r300_v1_smoke"
E3_OUT = "work/runs/e3_r300_v1"
E3_CP = f"{E3_OUT}/step-0300"
# Không có --stop-after200. Bước đích đầy đủ lấy300 từ YAML.
print(E3_ARGS, E3_SMOKE, E3_CP)

Sau ngắt runtime: khôi phục ZIP mới nhất của đúng run trước khi resume. Cell này an toàn khi không có ZIP; nó chỉ chuẩn bị, không train.

In [ ]:
from src.checkpoints import restore_run_archive, verify_checkpoint
for folder in (E3_SMOKE, E3_OUT):
    archive = Path(BACKUP) / f"{Path(folder).name}__latest.zip"
    if archive.exists():
        restore_run_archive(archive, folder)

def latest_checkpoint(folder):
    candidates = sorted(p for p in Path(folder).glob("step-*") if (p / "complete.json").is_file())
    return verify_checkpoint(candidates[-1], require_resume=True) if candidates else None

In [ ]:
marker = Path(E3_SMOKE) / "smoke_pass.json"
if marker.exists():
    smoke_result = json.loads(marker.read_text())
    assert smoke_result["resume_smoke_pass"] and smoke_result["resume_state_format"] == 1
    print("Giữ smoke pause/resume đã đạt:", smoke_result)
else:
    cp = latest_checkpoint(E3_SMOKE)
    if cp is None:
        run("-m", "src.run", "train", *E3_ARGS, "--out", E3_SMOKE,
            "--smoke", "--smoke-steps", "8", "--pause-at", "4", "--backup", BACKUP)
        cp = latest_checkpoint(E3_SMOKE)
    assert cp is not None
    run("-m", "src.run", "train", *E3_ARGS, "--out", E3_SMOKE,
        "--smoke", "--smoke-steps", "8", "--resume-from", str(cp), "--backup", BACKUP)
    print(marker.read_text())

### Xem số tham số kiến trúc thực đo
Đếm trên base model đã nạp ở Colab, trước gắn LoRA. Params4bit dùng shape gốc của quant_state, không lấy numel uint8 đã pack. Không cần nạp thêm model cho cell này.

In [ ]:
architecture = json.loads(Path(E3_SMOKE, "architecture.json").read_text())
print(json.dumps(architecture, indent=2))
(RESULTS / "analysis").mkdir(exist_ok=True)
shutil.copy2(Path(E3_SMOKE, "architecture.json"), RESULTS / "analysis/architecture.json")

## 3. Full E3 đủ300 — hoặc resume đúng run mới
Chỉ chạy sau SMOKE PASS có resume_smoke_pass=true. Mỗi100 bước lưu đầy đủ state. ZIP rolling __latest.zip của đúng run được thay sau khi upload và checksum đạt; run cũ đã nộp không bị đụng tới. Cần chỗ trống Drive: checkpoint optimizer E3/E4 có thể khoảng1.5–2GB mỗi run, upload cần thêm một bản tạm. Log I/O có thể làm thời gian dài hơn lượt cũ. BACKUP FAILED không dừng train, nhưng phải cứu ZIP local trước khi reset.

In [ ]:
if (Path(E3_CP) / "complete.json").exists():
    print("Giữ E3 đủ300 đã hoàn thành.")
else:
    cp = latest_checkpoint(E3_OUT)
    args = ["--resume-from", str(cp)] if cp else []
    run("-m", "src.run", "train", *E3_ARGS, "--out", E3_OUT,
        "--smoke-dir", E3_SMOKE, "--backup", BACKUP, *args)
run_info = json.loads(Path(E3_CP, "run.json").read_text())
assert run_info["steps"] == 300 and not run_info["smoke"]
print(json.dumps(run_info, indent=2))

### Muốn chủ động nghỉ trước khi hết GPU
Không dùng stop-after để đổi target. Có thể thêm --pause-at200 vào lệnh full trước khi chạy; nó lưu state tại200 rồi dừng, target vẫn300. Lần sau bỏ --pause-at và dùng --resume-from checkpoint200. Không dùng legacy step200 cũ. Nếu process ngắt bất chợt, khôi phục __latest.zip rồi chạy lại cell full; tiếp tục từ checkpoint hoàn tất gần nhất, log session cũ vẫn giữ.

## 4. Val E3 mới — chọn đối chứng trước test
Chỉ gỡ lỗi/chọn nền trên val. Không chạm e3.json200 đã nộp. Không điều chỉnh cấu hình bằng test.

In [ ]:
E3_VAL = "work/predictions/e3_r300_v1_val.jsonl"
E3_VAL_SCORE = RESULTS / "validation/e3.json"
if not E3_VAL_SCORE.exists():
    run("-m", "src.run", "predict", "--checkpoint", E3_CP, "--split", "val", "--out", E3_VAL)
    run("-m", "src.run", "score", "--checkpoint", E3_CP, "--split", "val",
        "--pred", E3_VAL, "--name", "E3_r300_v1", "--out", str(E3_VAL_SCORE))
print(E3_VAL_SCORE.read_text())

In [ ]:
from src.results import choose_by_val
from src.colab_workflow import e4_overrides
BASE = choose_by_val(RESULTS / "validation")
BASE_CP = E3_CP if BASE == "e3" else f"work/runs/{BASE}/step-0300"
base_run = json.loads(Path(BASE_CP, "run.json").read_text())
assert base_run["steps"] == 300
base_cfg = json.loads(Path(BASE_CP, "config.json").read_text())
assert base_cfg["max_steps"] == 300 and base_cfg["train_fraction"] == 1.0 and "stop_after" not in base_cfg
E4_ARGS = ["--config", f"configs/{BASE}.yaml", "--fraction", "0.1", *e4_overrides(BASE_CP, f"configs/{BASE}.yaml")]
E4_SMOKE = f"work/runs/e4_data10_r300_v1_{BASE}_smoke"
E4_OUT = f"work/runs/e4_data10_r300_v1_{BASE}"
E4_CP = f"{E4_OUT}/step-0300"
print("E4 đối chứng:", BASE, BASE_CP, "args:", E4_ARGS)

## 5. E4 10% đủ300 — smoke pause/resume rồi full
Giữ mọi tham số khác như setup được chọn. Cùng300 optimizer steps, horizon300; chỉ fraction khác. Đối chứng100% dùng đúng checkpoint/prediction đã có. Nếu ngân sách GPU thấp, dừng sau lưu E3/val và làm E4 vào session khác; không giảm số bước.

In [ ]:
for folder in (E4_SMOKE, E4_OUT):
    archive = Path(BACKUP) / f"{Path(folder).name}__latest.zip"
    if archive.exists():
        restore_run_archive(archive, folder)
marker = Path(E4_SMOKE) / "smoke_pass.json"
if marker.exists():
    assert json.loads(marker.read_text())["resume_smoke_pass"]
    print(marker.read_text())
else:
    cp = latest_checkpoint(E4_SMOKE)
    if cp is None:
        run("-m", "src.run", "train", *E4_ARGS, "--out", E4_SMOKE,
            "--smoke", "--smoke-steps", "8", "--pause-at", "4", "--backup", BACKUP)
        cp = latest_checkpoint(E4_SMOKE)
    run("-m", "src.run", "train", *E4_ARGS, "--out", E4_SMOKE,
        "--smoke", "--smoke-steps", "8", "--resume-from", str(cp), "--backup", BACKUP)
    print(marker.read_text())

In [ ]:
if not (Path(E4_CP) / "complete.json").exists():
    cp = latest_checkpoint(E4_OUT)
    args = ["--resume-from", str(cp)] if cp else []
    run("-m", "src.run", "train", *E4_ARGS, "--out", E4_OUT,
        "--smoke-dir", E4_SMOKE, "--backup", BACKUP, *args)
assert json.loads(Path(E4_CP, "run.json").read_text())["steps"] == 300
print(Path(E4_CP, "run.json").read_text())

## 6. Test mới và evidence mở rộng
E0/E1/E2 giữ prediction/evidence đã nộp. Chỉ suy luận test cho E3 mới và E4 mới, sau khi chốt setup trên val. Không ghi đè evidence cũ. Nếu file prediction tồn tại nhưng score chưa có, dừng xem có đủ image_ids trước khi chạy lại; CLI không tự ghi đè.

In [ ]:
E3_TEST = "work/predictions/e3_r300_v1_test.jsonl"
if not (RESULTS / "evidence/e3.json").exists():
    run("-m", "src.run", "predict", "--checkpoint", E3_CP, "--split", "test", "--out", E3_TEST)
    run("-m", "src.run", "score", "--checkpoint", E3_CP, "--split", "test",
        "--pred", E3_TEST, "--name", "E3_r300_v1", "--out", str(RESULTS / "evidence/e3.json"))
E4_TEST = f"work/predictions/{Path(E4_OUT).name}_test.jsonl"
if not (RESULTS / "evidence/e4_data_size_0.1.json").exists():
    run("-m", "src.run", "predict", "--checkpoint", E4_CP, "--split", "test", "--out", E4_TEST)
    run("-m", "src.run", "score", "--checkpoint", E4_CP, "--split", "test",
        "--pred", E4_TEST, "--name", "E4_data_size_0.1_r300_v1",
        "--out", str(RESULTS / "evidence/e4_data_size_0.1.json"))
BASE_TEST = E3_TEST if BASE == "e3" else f"work/predictions/{BASE}_test.jsonl"
if not (RESULTS / "evidence/e4_data_size_1.0.json").exists():
    run("-m", "src.run", "score", "--checkpoint", BASE_CP, "--split", "test",
        "--pred", BASE_TEST, "--name", "E4_data_size_1.0_r300_v1",
        "--out", str(RESULTS / "evidence/e4_data_size_1.0.json"))

## 7. Backup artifact nhỏ và gửi log
Chạy cell này sau mỗi chặng quan trọng, không đợi hết toàn bộ notebook. Nó chỉ sao lưu report/extended và log nhỏ; checkpoint mới đã có ZIP rolling riêng. Save a copy in Drive cho notebook UI có output. Không đưa ZIP trọng số vào Git.

In [ ]:
extended_drive = DRIVE / "extended300"
extended_drive.mkdir(parents=True, exist_ok=True)
shutil.copytree(RESULTS, extended_drive / "report/extended", dirs_exist_ok=True)
if Path("work/logs").exists():
    shutil.copytree("work/logs", extended_drive / "logs", dirs_exist_ok=True)
for name in ("e3_r300_v1_smoke", "e3_r300_v1", Path(E4_OUT).name if "E4_OUT" in globals() else None):
    if name is None:
        continue
    source = Path("work/runs") / name
    if source.exists():
        target = extended_drive / "run_metadata" / name
        target.mkdir(parents=True, exist_ok=True)
        for file in source.iterdir():
            if file.is_file() and file.suffix in (".json", ".jsonl", ".txt"):
                shutil.copy2(file, target / file.name)
print(extended_drive)

In [ ]:
# Sau khi có đủ kết quả: xuất ZIP chỉ chứa JSON/log nhỏ để gửi về local.
# Không chứa checkpoint/model, không thay REPORT/evidence đã nộp.
from zipfile import ZipFile, ZIP_DEFLATED
artifact_zip = Path("work/extended300_results.zip")
with ZipFile(artifact_zip, "w", ZIP_DEFLATED) as z:
    for p in RESULTS.rglob("*"):
        if p.is_file():
            z.write(p, p.as_posix())
    for folder in (E3_OUT, E4_OUT if "E4_OUT" in globals() else E3_SMOKE):
        for p in Path(folder).glob("*"):
            if p.is_file() and p.suffix in (".json", ".jsonl", ".txt"):
                z.write(p, f"report/extended/run_metadata/{Path(folder).name}/{p.name}")
files.download(str(artifact_zip))

## 8. Hoàn thiện sau khi có số thật
Gửi log smoke pause/resume và architecture trước full, rồi run.json, val và evidence mới. Sau đối chiếu sẽ chuyển kết quả300 vào báo cáo chính, giữ lịch sử200, cập nhật hình/phân tích theo prediction mới và chạy toàn bộ tests/checker trước nộp lại. Không tự thay kết quả cũ ngay khi có một file mới; không gọi các cell chưa chạy là đã đạt. E5 chưa làm.